In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/data.pdf")
documents = loader.load()


c:\Users\HP\Desktop\Assistant_LLM_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
for doc in documents:
    print(doc.page_content)  
    print(doc.metadata) 

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter=RecursiveCharacterTextSplitter(
   chunk_size=800,  # Taille max d’un morceau
    chunk_overlap=100   #Texte répété entre chunks pour le contexte
)
chunks = text_splitter.split_documents(documents)
len(chunks)

529

In [10]:
all_chunks = []

for doc in documents:
    chunks = text_splitter.split_documents([doc])
    all_chunks.extend(chunks)
    
print(f"Nombre total de chunks créés : {len(all_chunks)}")


Nombre total de chunks créés : 529


In [ ]:
from dotenv import load_dotenv
import os

dotenv_path = os.path.join("..", "app", ".env")  

load_dotenv(dotenv_path=dotenv_path)
hf_token = os.getenv("Assistant_LLM_RAG")


In [28]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma 


embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"use_auth_token": hf_token}
    
)

persist_dir = "./chroma_db"

# vectordb = Chroma.from_documents(
#     documents=all_chunks,
#     embedding=embeddings_model,
#     persist_directory=persist_dir
# )

vectordb = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings_model
)


vectordb.persist()

retriever = vectordb.as_retriever(search_kwargs={"k": 3}) 


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 110.73it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
